In [1]:
import librosa
import librosa.display
import shutil
import numpy as np
import pandas as pd
import noisereduce as nr
import soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from dataclasses import dataclass
from collections import Counter

d:\TB-LATEST-CLONE\Tuberculosis_Classification_using_CoughSound\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dataclasses import dataclass
from pathlib import Path
import shutil
import re

import numpy as np
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm import tqdm
import noisereduce as nr


# --- 1. Hyperparameter Configuration Area ---
@dataclass
class CoughProcessorConfig:
    """
    Centralized Hyperparameters for tuning.
    Adjusting these values impacts the sensitivity and quality of segmentation.
    """
    sr: int = 16000                  # Sampling Rate
    top_db: int = 15                 # Threshold for silence detection (Lower = more sensitive)
    min_cough_duration: float = 0.15  # Minimum duration (sec) to be considered a cough
    rms_threshold: float = 0.05      # Minimum energy (linear RMS) to filter low-level noise
    zcr_threshold: float = 0.05      # Minimum ZCR to filter low-frequency hum / silence-like segments

    nr_prop_decrease: float = 0.8    # Noise reduction strength (0.0 to 1.0)

    output_segment_sec: float = 1.0  # Fixed length per extracted segment
    offset_sec: float = 0.02         # Pre-roll before detected start (sec)

    gap_threshold_sec: float = 0.3   # Max gap between segments to merge (sec)

    # For visualization features
    hop_length: int = 256

    # Toggle showing plots (useful if you don't want plot for every file)
    show_plots: bool = False

    # Sorting behavior
    sort_by_numeric_stem: bool = True  # True: 001,002,... / False: lexicographic


# --- 2. TB Cough Segmentation Processor Class ---
class TBCoughSegmentationProcessor:
    """
    Professional Audio Processor for TB Cough Detection Research.
    Modular design for scalability.
    """

    def __init__(self, input_path: str, processed_path: str, config: CoughProcessorConfig):
        self.input_path = Path(input_path)
        self.processed_path = Path(processed_path)
        self.cleaned_path = self.processed_path / "cleaned_audio"
        self.segments_path = self.processed_path / "segments_extracted"
        self.cfg = config

        self.clear_folder()
        self.processed_path.mkdir(parents=True, exist_ok=True)

        for p in [self.cleaned_path, self.segments_path]:
            p.mkdir(parents=True, exist_ok=True)

    def clear_folder(self):
        if self.processed_path.exists():
            try:
                shutil.rmtree(self.processed_path)
                print(f"🧹 Cleaned existing directory: {self.processed_path}")
            except Exception as e:
                print(f"Error removing folder {self.processed_path}: {e}")

    def load_audio(self, file_path: Path) -> np.ndarray:
        """Loads audio file with target sampling rate."""
        y, _ = librosa.load(file_path, sr=self.cfg.sr)
        return y

    def noise_reduction(self, y: np.ndarray) -> np.ndarray:
        """Applies stationary spectral subtraction to clean background hiss."""
        return nr.reduce_noise(
            y=y,
            sr=self.cfg.sr,
            stationary=True,
            prop_decrease=self.cfg.nr_prop_decrease
        )

    @staticmethod
    def _first_int_in_stem(stem: str) -> int | None:
        """Extract first integer found in a filename stem; None if not found."""
        m = re.search(r"\d+", stem)
        return int(m.group()) if m else None

    def get_audio_files_sorted(self) -> list[Path]:
        """Return .wav files sorted (numeric stem preferred)."""
        files = list(self.input_path.glob("*.wav"))

        if not self.cfg.sort_by_numeric_stem:
            return sorted(files, key=lambda p: p.name.lower())

        def sort_key(p: Path):
            n = self._first_int_in_stem(p.stem)
            # (0, n) for numeric stems -> come first; (1, name) for non-numeric -> after
            return (0, n, p.name.lower()) if n is not None else (1, float("inf"), p.name.lower())

        return sorted(files, key=sort_key)

    def merge_intervals(self, intervals: list, gap_threshold: int) -> list:
        """Merge segments separated by a very short silence (gap_threshold in samples)."""
        if len(intervals) <= 1:
            return intervals

        merged = []
        curr_start, curr_end = intervals[0]

        for next_start, next_end in intervals[1:]:
            gap = next_start - curr_end
            if gap <= gap_threshold:
                curr_end = next_end
            else:
                merged.append([curr_start, curr_end])
                curr_start, curr_end = next_start, next_end

        merged.append([curr_start, curr_end])
        return merged

    def _shade_intervals(self, ax, intervals, color, alpha, label=None):
        """Helper to shade time intervals on an axis."""
        sr = self.cfg.sr
        first = True
        for start, end in intervals:
            ax.axvspan(
                start / sr,
                end / sr,
                color=color,
                alpha=alpha,
                label=(label if first else None)
            )
            first = False

    def visualize_segmentation_with_thresholds(
        self,
        y: np.ndarray,
        file_stem: str,
        intervals_raw: list,
        intervals_merged: list,
        intervals_valid: list
    ):
        """Show waveform + RMS/ZCR with threshold lines (no file export)."""
        if not self.cfg.show_plots:
            return

        sr = self.cfg.sr
        hop = self.cfg.hop_length

        # Frame-wise features
        rms = librosa.feature.rms(y=y, hop_length=hop)[0]  # linear
        rms_db = librosa.amplitude_to_db(rms, ref=np.max)  # dB (<= 0)
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
        t_feat = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop)

        # Threshold lines (visual reference)
        top_db_line = np.max(rms_db) - self.cfg.top_db  # ~ -top_db when ref=max
        rms_thr_db = librosa.amplitude_to_db(np.array([self.cfg.rms_threshold]), ref=np.max)[0]

        fig, axes = plt.subplots(3, 1, figsize=(15, 8), sharex=True)

        # Panel 1: waveform + intervals
        ax0 = axes[0]
        librosa.display.waveshow(y, sr=sr, ax=ax0, color="gray", alpha=0.6)
        self._shade_intervals(ax0, intervals_raw, color="gold", alpha=0.10, label="raw split")
        self._shade_intervals(ax0, intervals_merged, color="deepskyblue", alpha=0.15, label="merged")
        self._shade_intervals(ax0, intervals_valid, color="cyan", alpha=0.30, label="valid (kept)")
        ax0.set_title(
            f"{file_stem} | raw={len(intervals_raw)} merged={len(intervals_merged)} valid={len(intervals_valid)}"
        )
        ax0.set_ylabel("Amplitude")
        ax0.legend(loc="upper right")

        # Hyperparams box
        text = (
            f"top_db={self.cfg.top_db}\n"
            f"gap_merge={self.cfg.gap_threshold_sec}s\n"
            f"min_dur={self.cfg.min_cough_duration}s\n"
            f"rms_thr={self.cfg.rms_threshold} (linear)\n"
            f"zcr_thr={self.cfg.zcr_threshold}\n"
            f"offset={self.cfg.offset_sec}s\n"
            f"out_len={self.cfg.output_segment_sec}s"
        )
        ax0.text(
            0.01, 0.98, text,
            transform=ax0.transAxes,
            va="top", ha="left",
            fontsize=9,
            bbox=dict(boxstyle="round", alpha=0.15)
        )

        # Panel 2: RMS(dB) + thresholds
        ax1 = axes[1]
        ax1.plot(t_feat, rms_db)
        ax1.axhline(top_db_line, linestyle="--", linewidth=1, label=f"top_db line ({top_db_line:.1f} dB)")
        ax1.axhline(rms_thr_db, linestyle=":", linewidth=1, label=f"rms_threshold (~{rms_thr_db:.1f} dB)")
        self._shade_intervals(ax1, intervals_valid, color="cyan", alpha=0.12)
        ax1.set_ylabel("RMS (dB, ref=max)")
        ax1.legend(loc="lower right")

        # Panel 3: ZCR + threshold
        ax2 = axes[2]
        ax2.plot(t_feat, zcr)
        ax2.axhline(self.cfg.zcr_threshold, linestyle="--", linewidth=1,
                    label=f"zcr_threshold ({self.cfg.zcr_threshold})")
        self._shade_intervals(ax2, intervals_valid, color="cyan", alpha=0.12)
        ax2.set_ylabel("ZCR")
        ax2.set_xlabel("Time (s)")
        ax2.legend(loc="upper right")

        plt.tight_layout()
        plt.show()
        plt.close(fig)

    def segment_audio_event(self, y: np.ndarray, file_stem: str, file_folder: Path) -> int:
        """Detect, filter, visualize, and extract cough segments."""

        # 1) raw split
        intervals_raw = librosa.effects.split(y, top_db=self.cfg.top_db)

        # 2) merge
        gap_threshold_samples = int(self.cfg.gap_threshold_sec * self.cfg.sr)
        intervals_merged = self.merge_intervals(intervals_raw, gap_threshold_samples)

        # 3) filter
        valid_intervals = []
        min_samples = int(self.cfg.min_cough_duration * self.cfg.sr)

        for start, end in intervals_merged:
            if (end - start) < min_samples:
                continue

            segment_rms = float(np.max(librosa.feature.rms(y=y[start:end])))
            if segment_rms < self.cfg.rms_threshold:
                continue

            segment_zcr = float(np.mean(librosa.feature.zero_crossing_rate(y[start:end])))
            if segment_zcr < self.cfg.zcr_threshold:
                continue

            valid_intervals.append([start, end])

        # visualize
        self.visualize_segmentation_with_thresholds(
            y=y,
            file_stem=file_stem,
            intervals_raw=intervals_raw,
            intervals_merged=intervals_merged,
            intervals_valid=valid_intervals
        )

        # 4) extract fixed-length segments
        target_len_samples = int(self.cfg.output_segment_sec * self.cfg.sr)
        offset_samples = int(self.cfg.offset_sec * self.cfg.sr)

        for i, (start_idx, end_idx) in enumerate(valid_intervals):
            start_seg = max(0, start_idx - offset_samples)
            end_seg = start_seg + target_len_samples
            segment_y = y[start_seg:min(end_seg, len(y))]

            if len(segment_y) != target_len_samples:
                segment_y = librosa.util.fix_length(segment_y, size=target_len_samples)

            sf.write(file_folder / f"{file_stem}-{i+1:02d}.wav", segment_y, self.cfg.sr)

        return len(valid_intervals)

    def run_pipeline(self):
        """Execution orchestrator."""
        audio_files = self.get_audio_files_sorted()
        print(f"🚀 Initializing Pipeline for {len(audio_files)} files...")

        for audio in tqdm(audio_files):
            try:
                file_stem = audio.stem
                y = self.load_audio(audio)
                y_denoised = self.noise_reduction(y)
                y_norm = librosa.util.normalize(y_denoised)

                sf.write(self.cleaned_path / f"cleaned_{file_stem}.wav", y_norm, self.cfg.sr)
                count = self.segment_audio_event(y_norm, file_stem, self.segments_path)

                tqdm.write(f"  [SUCCESS] {audio.name}: Found {count} segments.")
            except Exception as e:
                print(f"  [ERROR] Processing {audio.name} failed: {e}")

        print(f"\n✅ Pipeline Complete! Output: {self.processed_path}")


if __name__ == "__main__":
    INPUT_DIR = "../raw_audio"
    OUTPUT_BASE_DIR = "../processed_audio"

    processor = TBCoughSegmentationProcessor(
        input_path=INPUT_DIR,
        processed_path=OUTPUT_BASE_DIR,
        config=CoughProcessorConfig()
    )

    processor.run_pipeline()


🚀 Initializing Pipeline for 130 files...


  0%|          | 0/130 [00:00<?, ?it/s]

  1%|          | 1/130 [00:09<19:23,  9.02s/it]

  [SUCCESS] 001_02.wav: Found 14 segments.


  2%|▏         | 2/130 [00:09<08:26,  3.96s/it]

...

  [SUCCESS] 016_08.wav: Found 3 segments.
  [SUCCESS] 016_09.wav: Found 3 segments.


 99%|█████████▉| 129/130 [00:32<00:00,  7.35it/s]

  [SUCCESS] 016_10.wav: Found 2 segments.
  [SUCCESS] 016_11.wav: Found 3 segments.


100%|██████████| 130/130 [00:32<00:00,  3.96it/s]

  [SUCCESS] 016_12.wav: Found 4 segments.

✅ Pipeline Complete! Output: ..\processed_audio


In [3]:
def count_segments_to_df(folder_path):
    path = Path(folder_path)
    files = [f.name.split('-')[0] for f in path.glob("*.wav")]
    df = pd.DataFrame(files, columns=['File'])
    df_counts = df['File'].value_counts().sort_index().reset_index()
    df_counts.columns = ['File', 'Count']
    return df_counts

TARGET_MANUAL = '../Data/ManualSegmentation'
TARGET_AUTOMATION = '../processed_audio/segments_extracted'

df_manual_segmentation = count_segments_to_df(TARGET_MANUAL)
df_automation_segmentation = count_segments_to_df(TARGET_AUTOMATION)
mn_count = df_manual_segmentation['Count'].sum()
at_count = df_automation_segmentation['Count'].sum()

print(f"Total Segments (Manual): {mn_count}")
print(f"Total Segments (Automation): {at_count}")
relative_error = ((abs(at_count - mn_count)) / at_count) * 100
accuracy = max(0, 100 - relative_error)
print(f"Total Matching % (Accuracy): {accuracy:.2f}%")
print(f"Relative_error: {relative_error:.2f}%")
df_compare = pd.merge(
    df_manual_segmentation, 
    df_automation_segmentation, 
    on='File', 
    how='outer',             
    suffixes=('_manual', '_auto') 
)
df_compare = df_compare.fillna(0)
df_compare['Difference'] = (df_compare['Count_auto'] - df_compare['Count_manual'])

df_compare.to_csv('segmentation_comparison.csv', index=False)

Total Segments (Manual): 512
Total Segments (Automation): 582
Total Matching % (Accuracy): 87.97%
Relative_error: 12.03%


In [4]:
def count_segments_to_df(folder_path):
    path = Path(folder_path)
    files = [f.name.split('-')[0] for f in path.glob("*.wav")]
    df = pd.DataFrame(files, columns=['File'])
    df_counts = df['File'].value_counts().sort_index().reset_index()
    df_counts.columns = ['File', 'Count']
    return df_counts

TARGET_MANUAL = '../Data/ManualSegmentation'
TARGET_AUTOMATION = '../processed_audio/segments_extracted'

manual_path = Path(TARGET_MANUAL)
auto_path = Path(TARGET_AUTOMATION)
print(f"Manual path: {manual_path.resolve()}")
print(f"Automation path: {auto_path.resolve()}")

df_manual_segmentation = count_segments_to_df(TARGET_MANUAL)
df_automation_segmentation = count_segments_to_df(TARGET_AUTOMATION)
mn_count = df_manual_segmentation['Count'].sum()
at_count = df_automation_segmentation['Count'].sum()

print(f"Total Segments (Manual): {mn_count}")
print(f"Total Segments (Automation): {at_count}")
if mn_count > 0:
    relative_error = (abs(at_count - mn_count) / mn_count) * 100
    accuracy = max(0, 100 - relative_error)
    print(f"Total Matching % (Accuracy): {accuracy:.2f}%")
    print(f"Relative_error: {relative_error:.2f}%")
else:
    print("Total Matching % (Accuracy): N/A (manual segment total is 0)")
    print("Relative_error: N/A (manual segment total is 0)")

df_compare = pd.merge(
    df_manual_segmentation, 
    df_automation_segmentation, 
    on='File', 
    how='outer',             
    suffixes=('_manual', '_auto') 
)
df_compare = df_compare.fillna(0)
df_compare[['Count_manual', 'Count_auto']] = df_compare[['Count_manual', 'Count_auto']].astype(int)

df_compare['Error'] = df_compare['Count_auto'] - df_compare['Count_manual']
mae = df_compare['Error'].abs().mean() if not df_compare.empty else 0.0
print(f"Mean Absolute Error (MAE): {mae:.2f} segments/file")

over_segments = int(df_compare.loc[df_compare['Error'] > 0, 'Error'].sum())
under_segments = int((-df_compare.loc[df_compare['Error'] < 0, 'Error']).sum())
print(f"Segments Over-detected (Total): {over_segments}")
print(f"Segments Under-detected (Total): {under_segments}")

df_compare.to_csv('segmentation_comparison.csv', index=False)

Manual path: D:\TB-LATEST-CLONE\Tuberculosis_Classification_using_CoughSound\Data\ManualSegmentation
Automation path: D:\TB-LATEST-CLONE\Tuberculosis_Classification_using_CoughSound\processed_audio\segments_extracted
Total Segments (Manual): 512
Total Segments (Automation): 582
Total Matching % (Accuracy): 86.33%
Relative_error: 13.67%
Mean Absolute Error (MAE): 0.71 segments/file
Segments Over-detected (Total): 81
Segments Under-detected (Total): 11
